# 04 - Estado de Cuenta y Portafolio

Este notebook prueba las funcionalidades de cuenta y portafolio de la API de IOL.

## Funcionalidades:
- Estado de cuenta (saldos en pesos y dolares)
- Portafolio de inversiones por pais
- Operaciones con filtros
- Detalle de operaciones

**Nota:** Requiere credenciales validas de IOL en el archivo `.env`

## Configuracion Inicial

In [ ]:
import sys
import os
from datetime import datetime, timedelta
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('../..'))

from pyIol import (
    IOLClient, IOLAPIError,
    EstadoCuenta, Cuenta, Saldo,
    Portafolio, Activo, TituloPortafolio,
    Operacion, OperacionDetalle, OperationStates, Countries
)
print("Librerias importadas correctamente")
print(f"Fecha y hora: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
load_dotenv('../../.env')
USERNAME = os.getenv('IOL_USERNAME', 'tu_usuario_iol')
PASSWORD = os.getenv('IOL_PASSWORD', 'tu_password_iol')

if USERNAME == "tu_usuario_iol":
    print("ADVERTENCIA: Configura las credenciales en .env")
else:
    print(f"Credenciales configuradas - Usuario: {USERNAME}")

In [ ]:
try:
    client = IOLClient(USERNAME, PASSWORD)
    print("Cliente IOL creado correctamente")
except Exception as e:
    print(f"Error al crear cliente: {e}")
    client = None

## 1. Estado de Cuenta

In [ ]:
# Estado de cuenta
if client:
    try:
        print("Obteniendo estado de cuenta...")
        estado = client.get_account_status()
        
        if estado and estado.cuentas:
            print(f"\nTotal de cuentas: {len(estado.cuentas)}")
            
            for cuenta in estado.cuentas:
                print(f"\nCuenta: {cuenta.numero} ({cuenta.tipo})")
                print(f"  Moneda: {cuenta.moneda}")
                print(f"  Saldo: ${cuenta.saldo:,.2f} | Disponible: ${cuenta.disponible:,.2f}")
                
                if cuenta.saldos:
                    for saldo in cuenta.saldos:
                        print(f"  Saldo {saldo.liquidacion}: ${saldo.disponible:,.2f} disponible")
                        print(f"    Comprometido: ${saldo.comprometido:,.2f}")
    except Exception as e:
        print(f"Error: {e}")

## 2. Portafolio de Inversiones

In [ ]:
# Portafolio
if client:
    try:
        print("Obteniendo portafolio de Argentina...")
        portafolio = client.get_portfolio(Countries.ARGENTINA)
        
        if portafolio:
            print(f"\nPais: {portafolio.pais}")
            print(f"Total valorizado: ${portafolio.total_valorizado:,.2f}")
            print(f"Ganancia total: ${portafolio.total_ganancia:,.2f}")
            
            # Usar la propiedad todos_los_titulos para obtener lista
            titulos = portafolio.todos_los_titulos
            if titulos:
                print(f"\nTotal de titulos: {len(titulos)}\n")
                for titulo in titulos[:10]:  # Mostrar primeros 10
                    print(f"  {titulo.simbolo} ({titulo.tipo}): {titulo.cantidad} unidades")
                    print(f"    Precio: ${titulo.ultimo_precio:,.2f} | PPC: ${titulo.ppc:,.2f}")
                    print(f"    Valorizado: ${titulo.valorizado:,.2f}")
                    print(f"    Ganancia: ${titulo.ganancia_dinero:,.2f} ({titulo.ganancia_porcentaje:+.2f}%)")
                if len(titulos) > 10:
                    print(f"\n  ... y {len(titulos) - 10} titulos mas")
            else:
                print("No hay titulos en el portafolio")
        else:
            print("Portafolio vacio o no disponible")
    except Exception as e:
        import traceback
        print(f"Error: {e}")
        traceback.print_exc()

## 3. Operaciones

In [ ]:
# Operaciones recientes
if client:
    try:
        print("Obteniendo operaciones recientes...")
        
        # Ultimos 30 dias
        fecha_hasta = datetime.now()
        fecha_desde = fecha_hasta - timedelta(days=30)
        
        operaciones = client.get_operations(
            estado=OperationStates.ALL,
            fecha_desde=fecha_desde,
            fecha_hasta=fecha_hasta
        )
        
        if operaciones:
            print(f"\nTotal de operaciones: {len(operaciones)}")
            
            for op in operaciones[:5]:
                print(f"\nOperacion #{op.numero}")
                print(f"  Tipo: {op.tipo} | Estado: {op.estado}")
                print(f"  Simbolo: {op.simbolo}")
                precio_str = f"${op.precio:,.2f}" if op.precio is not None else "N/A"
                print(f"  Cantidad: {op.cantidad or 'N/A'} @ {precio_str}")
                print(f"  Fecha: {op.fecha_ordenado}")
        else:
            print("No hay operaciones en el periodo")
    except Exception as e:
        print(f"Error: {e}")

In [ ]:
# Operaciones pendientes
if client:
    try:
        print("Obteniendo operaciones pendientes...")
        
        pendientes = client.get_operations(estado=OperationStates.PENDING)
        
        if pendientes:
            print(f"Operaciones pendientes: {len(pendientes)}")
            for op in pendientes:
                print(f"  #{op.numero}: {op.tipo} {op.simbolo} x{op.cantidad}")
        else:
            print("No hay operaciones pendientes")
    except Exception as e:
        print(f"Error: {e}")

## Metodos RAW Disponibles

In [ ]:
print("METODOS RAW DE CUENTA Y PORTAFOLIO")
print("="*50)
print("""
Para obtener respuestas en formato JSON crudo:

- client.get_account_status_raw()
- client.get_portfolio_raw(pais)
- client.get_operations_raw(estado, fecha_desde, fecha_hasta, ...)
- client.get_operation_detail_raw(numero_operacion)

Estos metodos retornan el JSON exacto de la API de IOL.
""")

## Limpieza

In [ ]:
if client:
    try:
        client.close()
        print("Cliente IOL cerrado correctamente")
    except Exception as e:
        print(f"Error al cerrar cliente: {e}")